In [1]:
"""
🔹 샘플링 (Negative Sampling)
여전히 필요함 ❗
E5를 써도 여전히 100만 개 전체 책 임베딩을 매번 비교하면 터짐.
E5는 “임베딩 품질”을 높일 뿐, “계산 효율”은 해결 안 해줌.
→ 따라서 여전히 BPR-style이나 sampled softmax로 학습해야 함.

E5 → 아이템 의미 표현 (고정)
SASRec → 유저 시퀀스 패턴 학습
BPR Loss → 효율적 학습 (샘플링 기반)
FAISS → 추천 시 빠른 검색
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import get_cosine_schedule_with_warmup

In [2]:
"""
원랜 다른 버전...(오리지날)으로 하려햇는데.. 
self.item_embedding = nn.Embedding(n_items, hidden_dim) 이 코드에서...
아이템 개(n_items) × 임베딩 차원(hidden_dim)짜리 float32 행렬을 램에 만드느라 계산 터진다네?

근데 우리 가만 생각해보면 책 정보를 이미 E5로 임베딩해놨었잖아????? 걍 그걸 쓰도록 새롭게 모델 만들어보자구
"""
class SASRec(nn.Module):
    def __init__(self, input_dim=384, hidden_dim=128, n_layers=2, n_heads=2, max_len=150, dropout=0.2, pad_idx=None, out_dim=384):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.input_proj = nn.Linear(input_dim, hidden_dim)  # E5 dim -> hidden_dim
        self.pos_embedding = nn.Embedding(max_len, hidden_dim)
        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=n_heads, dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=n_layers)
        self.dropout = nn.Dropout(dropout)
        self.pad_idx = pad_idx
        # output projection (128 -> 384), trainable and part of model
        self.user_proj = nn.Linear(hidden_dim, out_dim)

    def forward(self, seq_embs, src_key_padding_mask=None):
        # seq_embs: (B, L, input_dim)
        B, L, D = seq_embs.size()
        # 이거 그 차원을 384로 하니까 너무 과적합되는거 같아서 데이터는 적은데...
        seq = self.input_proj(seq_embs)  # 이제 차원이 [B, L, hidden_dim]
        # positions: shape (B, L)
        positions = torch.arange(L, device=seq.device).unsqueeze(0).expand(B, L)
        seq = seq + self.pos_embedding(positions)
        seq = self.dropout(seq)
        # encoder supports src_key_padding_mask: True for positions to be masked
        out = self.encoder(seq, src_key_padding_mask=src_key_padding_mask)  # [B, L, hidden_dim]
        return out  # keep full sequence outputs
    """
    get_user_embedding()이 하는 일
    
    get_user_embedding()은 SASRec의 핵심 부분으로,
    입력된 시퀀스 임베딩(user_seq_embs: [B, L, D])을 Transformer 블록에 통과시켜서
    “이 사용자가 최근 어떤 책을 봤고, 앞으로 어떤 책을 볼 가능성이 높은가”를 반영한
    하나의 latent user embedding을 만드는 함수야.
    
    즉, user_emb = self.user_tower.get_user_embedding(user_seq_embs)는
    ➡️ “E5 임베딩 시퀀스를 Transformer로 요약해서 user preference vector를 뽑는 단계”야.
    
    ⚙️ 만약 get_user_embedding()을 안 쓰고 그냥 평균하거나 그대로 넣는다면?
    
    단순히 “유저가 본 책들의 평균 의미”밖에 안 돼.
    
    순서 정보(시간적 패턴)와 **행동 기반 추천 패턴(‘SASRec의 강점’)**이 사라짐.
    
    즉, Transformer를 안 쓰는 Two-Tower 모델이 되어버려.
    """
    # def get_user_embedding(self, seq_embs):
    #     seq_out = self.forward(seq_embs)
    #     return seq_out[:, -1, :]  # 마지막 아이템 기준 user representation
    def get_user_embedding(self, seq_embs, seq_indices=None):
        """
        seq_embs: (B, L, input_dim)  (raw item embeddings)
        seq_indices: (B, L) long tensor containing item indices (only needed to build mask)
        returns: (B, out_dim) projected user vectors
        """
        # build padding mask if pad_idx known and seq_indices provided
        mask = None
        if seq_indices is not None and self.pad_idx is not None:
            # src_key_padding_mask expects True at positions that should be masked
            mask = (seq_indices == self.pad_idx)  # [B, L], bool

        seq_out = self.forward(seq_embs, src_key_padding_mask=mask)  # [B, L, hidden]
        # pooling: take last non-pad token output (recommended for next-item)
        if seq_indices is not None and self.pad_idx is not None:
            # compute lengths
            not_pad = (~(seq_indices == self.pad_idx))  # True where real
            lengths = not_pad.sum(dim=1)  # [B]
            lengths = torch.clamp(lengths - 1, min=0)  # last valid index per example
            # gather last vectors
            idx = lengths.view(-1,1,1).expand(-1,1,seq_out.size(-1))  # [B,1,hidden]
            last_vec = seq_out.gather(1, idx).squeeze(1)  # [B, hidden]
            user_hidden = last_vec
        else:
            # fallback: mean pooling
            user_hidden = seq_out.mean(dim=1)
        user_proj = self.user_proj(user_hidden)  # [B, out_dim]
        return user_proj

In [3]:
"""
1. Two-Tower Hybrid Recommender
유저 시퀀스와 아이템 임베딩을 각각 처리하는 구조

2. 그냥 SASRec output을 그대로 book_emb랑 dot product하면 의미 정보 반영 안 됨
→ 반드시 projection layer + 학습(loss) 을 통해 user_emb를 semantic space로 맞춰야 함
  ** 학습: projection + contrastive loss를 통해 두 임베딩 공간 align
즉, “책 의미를 반영하려면 학습 과정에서 E5 embedding을 정답(label)로 삼아야 한다”

3. 그냥 SASRec만 지나온 user_emb와 E5 book_emb는 처음엔 의미가 없지만,
Two-Tower 학습(BPR/contrastive)으로 projection layer가 학습되면서 dot product가 실제 추천 점수로 작동
즉, SASRec 자체가 “E5 embedding처럼 뽑아낸다”가 아니라,
학습 과정에서 latent user embedding을 item embedding space에 맞춰 조정한다고 보는 게 정확함

4. (semantic info를 user embedding에도 반영하고 싶다면)
방법1: SASRec 입력 시 book_id 대신 E5 embedding sequence 사용 → Transformer 기반 user tower
방법2: Two-Tower 학습 시 cross-attention or Bag-of-items attention
방법3: 현재 방식 그대로 유지, user embedding은 sequence 패턴만, item embedding은 semantic content만 사용 → 산업용 추천에서는 흔한 방식
"""
class TwoTowerRecommender(nn.Module):
    def __init__(self, 
                 sasrec_model,      # 이미 정의된 SASRec 모델
                 e5_dim=384,        # E5 embedding 차원(book_emb)
                 hidden_dim=128,    # SASRec output 차원(user_pref_emb)
                 proj_dim=128):     # projection 차원 (user_emb -> item_emb space)
        super().__init__()
        
        # ==== User Tower ====
        self.user_tower = sasrec_model  # SASRec 재활용
        self.user_proj = nn.Linear(hidden_dim, proj_dim)  # SASRec -> projection (item_emb 차원)
        
        # ==== Item Tower ====
        self.item_proj = nn.Linear(e5_dim, proj_dim)      # E5 -> projection (item_emb 차원)
        
        self.dropout = nn.Dropout(0.1)

    def forward(self, user_seq_embs, book_embs, neg_book_embs=None):
        """
        user_seqs: [batch, seq_len]   (book_id 시퀀스)
        user_seq_embs: [batch, seq_len, e5_dim]  (E5 시퀀스)
        book_embs: [batch, e5_dim]    (positive book embedding)
        neg_book_embs: [batch, n_neg, e5_dim] (negative embeddings optional)
        """
        # 1. User embedding
        # get_user_embedding은 내가 아까 SASRec에서 정의한 함수인데... 모델 가져와서 쓰려면 바꿔야함
        # uesr_seqs_embs라는 패턴을 transformer로 요약하는 과정이래 SASRec를 통해서
        user_emb = self.user_tower.get_user_embedding(user_seq_embs)   # [B, hidden_dim]
        user_emb = self.dropout(self.user_proj(user_emb))          # [B, proj_dim]

        # 2. Positive book embedding
        pos_emb = self.dropout(self.item_proj(book_embs))          # [B, proj_dim]

        # 3. Compute scores (dot product)
        pos_scores = torch.sum(user_emb * pos_emb, dim=-1)         # [B]

        """
        여기 부분 잘 모르겠음
        """
        if neg_book_embs is not None:
            # negative embeddings projection
            B, n_neg, e5_dim = neg_book_embs.shape
            neg_embs = self.item_proj(neg_book_embs.view(-1, e5_dim))  # [B*n_neg, proj_dim]
            neg_embs = self.dropout(neg_embs).view(B, n_neg, -1)      # [B, n_neg, proj_dim]
            
            # dot product with user_emb
            neg_scores = torch.bmm(neg_embs, user_emb.unsqueeze(-1)).squeeze(-1)  # [B, n_neg]
            return pos_scores, neg_scores
        
        return pos_scores

    def recommend(self, user_seq_embs, all_book_embs, topk=10):
        """
        top-k 추천용
        all_book_embs: [num_books, e5_dim]
        """
        user_emb = self.user_tower.get_user_embedding(user_seq_embs)  # [B, hidden_dim]
        user_emb = self.user_proj(user_emb)                       # [B, proj_dim]
        all_book_embs_proj = self.item_proj(all_book_embs)        # [num_books, proj_dim]

        # 유사도 계산 (dot product)
        scores = torch.matmul(user_emb, all_book_embs_proj.T)     # [B, num_books]
        topk_scores, topk_idx = torch.topk(scores, k=topk, dim=1)
        return topk_idx, topk_scores

In [4]:
# =========================================
# 0️⃣ 기본 설정
# =========================================
PAD_ID = -1 # 모델에서 padding_idx랑 일치시켜야 함
EMB_DIM = 384  # E5 임베딩 차원

# book_emb_dict에 PAD_ID embedding 추가
book_emb_dict = np.load("book_emb_dict.npy", allow_pickle=True).item()
book_emb_dict[PAD_ID] = np.zeros(EMB_DIM, dtype=np.float32)

DATA_PATH = './data/user_book_sequence.parquet'
df_user = pd.read_parquet(DATA_PATH)

# 시퀀스 전처리: 없는 책은 PAD_ID로 대체
df_user["book_seq_padded"] = df_user["book_seq"].apply(
    lambda seq: [b if b in book_emb_dict else PAD_ID for b in seq]
)

"""
SASRec, BERT4Rec, GRU4Rec, 이런 시퀀스 기반 모델들은
기본적으로 “임베딩 테이블(index 기반)”을 입력으로 받음

즉, embedding_table[i] → i번째 아이템의 벡터
각 아이템(Book)을 “정수 index”로 표현해야 함
"""
# SASRec 입력용 index 변환을 위한 매핑
book_ids = sorted(book_emb_dict.keys())
book2idx = {b: i for i, b in enumerate(book_ids)}
idx2book = {i: b for i, b in enumerate(book_ids)}

# torch 텐서로 변환
book_emb_matrix = torch.tensor(
    np.stack([book_emb_dict[b] for b in book_ids]),
    dtype=torch.float32  # float16로 메모리 절감은 나중에
)  # (num_books, 384)

# dict 삭제 (RAM 확보)
del book_emb_dict
import gc; gc.collect()

0

In [5]:
# =========================================
# user sequences & positive book embeddings
# ① 유저별 시퀀스를 인덱스로 바꾸고
# ② 마지막 책을 positive target으로 뽑은 다음
# ③ 그 책들의 임베딩(pos_book_embs)을 텐서로 만드는 것
# =========================================
max_len = 100
user_seqs_list = []
pos_book_ids_list = []

for seq in df_user["book_seq_padded"]:
    seq = list(seq)
    if len(seq) == 0:
        user_seqs_list.append([book2idx[PAD_ID]] * max_len)
        pos_book_ids_list.append(book2idx[PAD_ID])
        continue

    pos_book_id = seq[-1]
    pos_book_ids_list.append(book2idx.get(pos_book_id, book2idx[PAD_ID]))

    pad_len = max(0, max_len - len(seq))
    seq_idx = [book2idx.get(b, book2idx[PAD_ID]) for b in seq[-max_len:]]
    seq_idx = [book2idx[PAD_ID]] * pad_len + seq_idx
    user_seqs_list.append(seq_idx)

# 텐서 변환
user_seqs_tensor = torch.tensor(user_seqs_list, dtype=torch.long)
pos_book_ids_tensor = torch.tensor(pos_book_ids_list, dtype=torch.long)

In [6]:
"""
Offline Negative Sampling
→ “샘플링 방식이 복잡하거나 비모델 의존적일 때” (예: popularity-based, sequence-aware 등)
→ 즉, 학습 중에 계산하기엔 너무 무거울 때

고정된 Negative Pool을 쓰는 경우
→ 학습 중에 랜덤이 아니라, 특정 negative 후보군을 유지하고 싶을 때

이럴때만 dataset에서 neg sampling을 하니까 일단은 빼도록 하자

# **neg sampling**
# 같은 pos pool에서 랜덤 선택 ㄴㄴ
# 전체 `book_emb_matrix`에서 샘플링 (훨씬 일반적) 하도록 해야하는데 나중에..
"""
class UserBookDataset(Dataset):
    def __init__(self, user_seqs, pos_book_ids, book_emb_matrix):
        """
        user_seqs: list or tensor (num_users, seq_len)
        pos_book_ids: tensor (num_users,)
        book_emb_matrix: torch.Tensor (num_books, emb_dim)
        """
        self.user_seqs = user_seqs
        
        # self.pos_book_embs = pos_book_embs -> init에서 직접 들고 있으면 메모리 터짐
        # `pos_book_ids`만 저장하고, 임베딩은 on-demand로 가져옴 |
        self.pos_book_ids = pos_book_ids

        self.book_emb_matrix = book_emb_matrix
        self.num_books = book_emb_matrix.size(0)
    
    def __len__(self):
        return len(self.user_seqs)

    def __getitem__(self, idx):
        # user sequence
        ''' 
        user_seq = torch.tensor(self.user_seqs[idx], dtype=torch.long)
        `user_seqs`가 이미 tensor면 그대로 사용 가능??????? 
        '''
        user_seq = self.user_seqs[idx]

        # positive book embedding
        pos_id = self.pos_book_ids[idx]
        # pos_emb = self.book_emb_matrix[pos_id]
        # 이거.... pos_emb를 가져오면서 collate_fn이 하는 역할을 침범하고 있대
        # collate_fn에서 batch 단위에서 pos/neg 임베딩 변환을 처리해야 하는데..

        return user_seq, pos_id

In [7]:
"""
Dataset --→ DataLoader --→ Model
            ㄴcollate_fn
"""
# 👇 collate_fn에서 배치 단위로 negative sampling
# 샘플링을 배치단위로 할거라서! 개별 인덱스 단위인 Dataset과는 다르게
# def collate_fn_with_neg(batch, book_emb_matrix, n_neg=5):
#     user_seqs, pos_ids = zip(*batch)
#     user_seqs = torch.stack(user_seqs)
#     pos_ids = torch.stack(pos_ids)

#     num_books = book_emb_matrix.size(0)
#     batch_size = len(batch)

#     user_seq_embs = book_emb_matrix[user_seqs]  # [B, L, D]
#     pos_embs = book_emb_matrix[pos_ids]         # [B, D]
#     neg_ids = torch.randint(0, num_books, (batch_size, n_neg)) # 배치 단위 랜덤 negative sampling
#     # 얜 완전 랜덤
#     neg_embs = book_emb_matrix[neg_ids]  # (B, n_neg, D)

#     return user_seq_embs, pos_embs, neg_embs
"""
1. 너무 쉬운 negative
완전히 랜덤으로 뽑으면 대부분의 negative가 positive와 거의 관계 없는 아이템
모델이 “positive > negative”를 금방 학습 → loss 급락
Valid loss가 0.001~0.01 수준으로 매우 낮게 나오는 이유

2. 학습 속도 급격 → 과적합 위험
데이터가 적으면 negative가 다양하지 않아, 몇 epoch만에 모델이 배치 내 negative에 최적화됨

=> 개선 방법 (Negative Sampling 전략)
1) Hard Negative Sampling
현재 모델이 positive와 dot-product가 높은 아이템 중 일부를 negative로 사용
“비슷한데 틀린” 아이템을 학습시키면 InfoNCE가 더 안정적

neg_ids = torch.randint(0, batch_size, (batch_size, n_neg))
neg_embs = pos_book_embs[neg_ids]

2) In-batch Negative
배치 내 다른 positive embedding을 negative로 사용 → 학습 안정적

# user_emb: [B, D], pos_emb: [B, D]
neg_embs = pos_emb[torch.randperm(pos_emb.size(0))]  # permute batch


3) Mix of Random + Hard Negative
절반은 random, 절반은 in-batch hard negative
모델이 쉽게 폭락하지 않고, valid에서도 안정적

4) Top-K Sampling
학습 초반에는 random, 중간 이후 epoch부터는 similarity가 높은 negative 선택
curriculum learning 느낌
"""

'\n1. 너무 쉬운 negative\n완전히 랜덤으로 뽑으면 대부분의 negative가 positive와 거의 관계 없는 아이템\n모델이 “positive > negative”를 금방 학습 → loss 급락\nValid loss가 0.001~0.01 수준으로 매우 낮게 나오는 이유\n\n2. 학습 속도 급격 → 과적합 위험\n데이터가 적으면 negative가 다양하지 않아, 몇 epoch만에 모델이 배치 내 negative에 최적화됨\n\n=> 개선 방법 (Negative Sampling 전략)\n1) Hard Negative Sampling\n현재 모델이 positive와 dot-product가 높은 아이템 중 일부를 negative로 사용\n“비슷한데 틀린” 아이템을 학습시키면 InfoNCE가 더 안정적\n\nneg_ids = torch.randint(0, batch_size, (batch_size, n_neg))\nneg_embs = pos_book_embs[neg_ids]\n\n2) In-batch Negative\n배치 내 다른 positive embedding을 negative로 사용 → 학습 안정적\n\n# user_emb: [B, D], pos_emb: [B, D]\nneg_embs = pos_emb[torch.randperm(pos_emb.size(0))]  # permute batch\n\n\n3) Mix of Random + Hard Negative\n절반은 random, 절반은 in-batch hard negative\n모델이 쉽게 폭락하지 않고, valid에서도 안정적\n\n4) Top-K Sampling\n학습 초반에는 random, 중간 이후 epoch부터는 similarity가 높은 negative 선택\ncurriculum learning 느낌\n'

In [8]:
def collate_fn_with_neg(batch, book_emb_matrix, n_neg=5, hard_ratio=0.5, pad_idx=None):
    """
    batch: list of (user_seq, pos_id)
    book_emb_matrix: [num_books, emb_dim]
    n_neg: negative sample 개수
    hard_ratio: in-batch hard negative 비율
    """
    user_seqs_idx, pos_ids = zip(*batch)
    # ensure tensors
    user_seqs_idx = torch.stack(user_seqs_idx).long()  # [B, L]
    pos_ids = torch.stack(pos_ids).long()              # [B]
    B, L = user_seqs_idx.shape

    # Embedding lookup: prefer nn.Embedding.from_pretrained(book_emb_matrix) and use indices
    # If book_emb_matrix is tensor passed here, indexing is OK but costly if then .to(device)
    user_seq_embs = book_emb_matrix[user_seqs_idx]  # [B, L, D]
    pos_embs = book_emb_matrix[pos_ids]             # [B, D]

    # negatives
    n_hard = int(n_neg * hard_ratio)
    n_rand = n_neg - n_hard
    neg_embs_list = []
    if n_hard > 0:
        # perm guaranteed non-equal by rotate
        perm = (torch.arange(B) + 1) % B
        hard_neg = pos_embs[perm][:, None, :].repeat(1, n_hard, 1)  # [B, n_hard, D]
        neg_embs_list.append(hard_neg)
    if n_rand > 0:
        num_books = book_emb_matrix.size(0)
        rand_ids = torch.randint(0, num_books, (B, n_rand), dtype=torch.long)
        rand_neg = book_emb_matrix[rand_ids]  # [B, n_rand, D]
        neg_embs_list.append(rand_neg)

    neg_embs = torch.cat(neg_embs_list, dim=1)  # [B, n_neg, D]
    return user_seq_embs, pos_embs, neg_embs, user_seqs_idx

In [9]:
"""
UserTower + ItemTower: SASRec + E5 투타워 구조

Negative Sampling: batch마다 랜덤 n_neg negative

Loss: BPR-style contrastive loss (pos > neg)

Projection: user_emb → E5 차원, item_emb → 동일 projection

Top-k 추천: precomputed E5 book_emb dot product로 빠르게 추천 가능
"""
# ===============================
# Hyperparameters
# ===============================
batch_size = 128
e5_dim = 384
hidden_dim = 128
proj_dim = 128
n_neg = 5
lr = 1e-3
weight_decay=1e-5
tau = 0.07
epochs = 20

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [10]:
# dataset = UserBookDataset(
#     user_seqs=user_seqs_tensor,
#     pos_book_ids=pos_book_ids_tensor,
#     book_emb_matrix=book_emb_matrix
# )
# dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

# --- 1️⃣ train / valid split ---
num_users = len(user_seqs_tensor)
split = int(num_users * 0.8)

train_user_seqs = user_seqs_tensor[:split]
train_pos_ids = pos_book_ids_tensor[:split]

valid_user_seqs = user_seqs_tensor[split:]
valid_pos_ids = pos_book_ids_tensor[split:]

# --- 2️⃣ dataset 각각 생성 ---
train_dataset = UserBookDataset(train_user_seqs, train_pos_ids, book_emb_matrix)
valid_dataset = UserBookDataset(valid_user_seqs, valid_pos_ids, book_emb_matrix)

# --- 3️⃣ dataloader 각각 생성 ---
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=lambda b: collate_fn_with_neg(b, book_emb_matrix, n_neg=5)
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=lambda b: collate_fn_with_neg(b, book_emb_matrix, n_neg=5)
)

In [11]:
def eval_sampled(user_embs, pos_embs, neg_embs, ks=(1,5,10)):
    # user_embs: [B, D], pos_embs: [B, D], neg_embs: [B, n_neg, D]
    # 모두 normalize되어 있다고 가정
    B = user_embs.size(0)
    n_neg = neg_embs.size(1)
    # sims: pos and neg
    pos_sim = torch.einsum('bd,bd->b', user_embs, pos_embs)  # [B]
    neg_sim = torch.einsum('bd,bnd->bn', user_embs, neg_embs)  # [B, n_neg]
    sims = torch.cat([pos_sim.unsqueeze(1), neg_sim], dim=1)  # [B, 1+n_neg]
    ranks = torch.argsort(sims, dim=1, descending=True)  # [B, 1+n_neg]

    recalls = {k: 0.0 for k in ks}
    ndcgs = {k: 0.0 for k in ks}
    mrr = 0.0

    for i in range(B):
        rank_list = ranks[i].cpu().numpy()
        # pos is index 0 originally
        # find position of 0
        pos_rank = int(np.where(rank_list == 0)[0][0]) + 1  # 1-based
        mrr += 1.0 / pos_rank
        for k in ks:
            recalls[k] += (1.0 if pos_rank <= k else 0.0)
            # ndcg (single relevant)
            ndcgs[k] += (1.0 / np.log2(pos_rank+1)) if pos_rank <= k else 0.0

    metrics = {'MRR': mrr/B}
    for k in ks:
        metrics[f'Recall@{k}'] = recalls[k]/B
        # normalize ndcg by ideal (which is 1/log2(1+1) if single relevant)
        metrics[f'NDCG@{k}'] = ndcgs[k]/B / (1.0/np.log2(2))
    return metrics

In [12]:
"""
1. InfoNCF pre-train
SASRec가 뽑아내는 임베딩 결과(user-item 요약)가 pos_book_emb와 유사하게 되도록!
=> user-item 간 semantic similarity를 학습시키는 과정
"""
# export CUDA_LAUNCH_BLOCKING=1

sasrec = SASRec(
    input_dim=e5_dim,
    hidden_dim=128,  # 이전 384 → 128
    n_layers=1,      # 이전 2 → 1
    n_heads=2,       # 그대로
    max_len=max_len,
    dropout=0.2
).to(device)

optimizer = optim.AdamW(sasrec.parameters(), lr=lr, weight_decay=weight_decay)

num_training_steps = len(train_loader) * epochs
num_warmup_steps = int(num_training_steps * 0.1)  # 10% warmup

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

for epoch in range(epochs):
    # =====================================================
    # 🔹 1) TRAIN PHASE
    # =====================================================
    sasrec.train()
    total_loss = 0.0

    for user_seq_embs, pos_book_embs, neg_book_embs, user_seqs_idx in tqdm(train_loader, desc=f"Epoch: {epoch+1}"):
        user_seq_embs = user_seq_embs.to(device)
        pos_book_embs = pos_book_embs.to(device)
        neg_book_embs = neg_book_embs.to(device)
        seq_indices = user_seqs_idx.to(device)

        # (1) user embedding (sequence → vector)
        user_emb = sasrec.get_user_embedding(user_seq_embs, seq_indices)  # [B, D]
        
        user_emb = F.normalize(user_emb, dim=-1)
        pos_book_embs = F.normalize(pos_book_embs, dim=-1)
        neg_book_embs = F.normalize(neg_book_embs, dim=-1)
        
        # (2) Similarity 계산
        pos_sim = torch.sum(user_emb * pos_book_embs, dim=-1) / tau                # [B]
        # neg_sim = torch.bmm(neg_book_embs, user_emb.unsqueeze(-1)).squeeze(-1) / tau  # [B, n_neg]
        neg_sim = torch.einsum('bd,bnd->bn', user_emb, neg_book_embs) / tau # [B,n_neg]

        # (3) InfoNCE Loss
        logits = torch.cat([pos_sim.unsqueeze(1), neg_sim], dim=1)  # [B, 1 + n_neg]
        labels = torch.zeros(len(pos_sim), dtype=torch.long, device=pos_sim.device)
        loss = F.cross_entropy(logits, labels)

        # (4) Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad() # 이번 batch에서 쌓인 gradient를 초기화
        total_loss += loss.item() * user_seq_embs.size(0)

    avg_train_loss = total_loss / len(train_loader.dataset)
    
    # =====================================================
    # 🔹 2) VALID PHASE
    # =====================================================
    sasrec.eval()
    total_valid_loss = 0.0

    """ --------------------------------------------------------------------- """
    all_metrics = []
    """ --------------------------------------------------------------------- """
    
    with torch.no_grad():
        for user_seq_embs, pos_book_embs, neg_book_embs, user_seqs_idx in valid_loader:
            user_seq_embs = user_seq_embs.to(device)
            pos_book_embs = pos_book_embs.to(device)
            neg_book_embs = neg_book_embs.to(device)
            seq_indices = user_seqs_idx.to(device)

            # (1) user embedding
            user_emb = sasrec.get_user_embedding(user_seq_embs, seq_indices)

            # (2) Normalize
            user_emb = F.normalize(user_emb, dim=-1)
            pos_book_embs = F.normalize(pos_book_embs, dim=-1)
            neg_book_embs = F.normalize(neg_book_embs, dim=-1)

            # (3) Similarity
            pos_sim = torch.sum(user_emb * pos_book_embs, dim=-1) / tau
            neg_sim = torch.einsum('bd,bnd->bn', user_emb, neg_book_embs) / tau

            # (4) InfoNCE Loss
            logits = torch.cat([pos_sim.unsqueeze(1), neg_sim], dim=1)
            labels = torch.zeros(len(pos_sim), dtype=torch.long, device=pos_sim.device)
            loss = F.cross_entropy(logits, labels)

            total_valid_loss += loss.item() * user_seq_embs.size(0)
    
            """ ------------------------------------------------------------- """
            # 여기에서 샘플링 기반 평가
            metrics = eval_sampled(user_emb, pos_book_embs, neg_book_embs, ks=(5,10))
            all_metrics.append(metrics)

        # 평균
        mean_metrics = {
            k: sum(m[k] for m in all_metrics)/len(all_metrics)
            for k in all_metrics[0].keys()
        }

    print(f"Recall@5={mean_metrics['Recall@5']:.4f}, NDCG@5={mean_metrics['NDCG@5']:.4f}, MRR={mean_metrics['MRR']:.4f}")
    """ --------------------------------------------------------------------- """

    avg_valid_loss = total_valid_loss / len(valid_loader.dataset)

    # =====================================================
    # 🔹 3) 결과 출력
    # =====================================================
    print(f"[Epoch {epoch+1}/{epochs}] "
          f"Train Loss: {avg_train_loss:.4f} | Valid Loss: {avg_valid_loss:.4f}")

Epoch: 1: 100%|████████████████████████████████████████████████████| 5228/5228 [00:50<00:00, 103.72it/s]


Recall@5=1.0000, NDCG@5=0.9939, MRR=0.9919
[Epoch 1/20] Train Loss: 0.8750 | Valid Loss: 0.3078


Epoch: 2: 100%|████████████████████████████████████████████████████| 5228/5228 [00:50<00:00, 104.37it/s]


Recall@5=1.0000, NDCG@5=0.9985, MRR=0.9980
[Epoch 2/20] Train Loss: 0.2002 | Valid Loss: 0.1009


Epoch: 3: 100%|████████████████████████████████████████████████████| 5228/5228 [00:49<00:00, 105.71it/s]


...

Recall@5=1.0000, NDCG@5=0.9987, MRR=0.9982
[Epoch 15/20] Train Loss: 0.0557 | Valid Loss: 0.0451


Epoch: 16: 100%|███████████████████████████████████████████████████| 5228/5228 [00:48<00:00, 108.14it/s]


Recall@5=1.0000, NDCG@5=0.9987, MRR=0.9983
[Epoch 16/20] Train Loss: 0.0556 | Valid Loss: 0.0447


Epoch: 17:   2%|█                                                   | 105/5228 [00:00<00:47, 107.53it/s]


KeyboardInterrupt: 

In [ ]:
# user_seqs = train_user_seqs
# seq_lengths = [len(seq) for seq in user_seqs]  # user_seqs: list of lists
seq_lengths = [len(seq) for seq in df_user["book_seq_padded"]]

import matplotlib.pyplot as plt

plt.hist(seq_lengths, bins=30)
plt.xlabel("Sequence length")
plt.ylabel("Number of users")
plt.title("User sequence length distribution")
plt.show()

print("Min/Max/Mean/Median sequence length:", min(seq_lengths), max(seq_lengths), np.mean(seq_lengths), np.median(seq_lengths))
print("Average events per user:", np.mean(seq_lengths))

In [ ]:
from collections import Counter

all_items = [item for seq in user_seqs_list for item in seq if item != book2idx[PAD_ID]]
item_counts = Counter(all_items)

# 히스토그램
plt.figure(figsize=(8,5))
plt.hist(list(item_counts.values()), bins=50, color='skyblue', edgecolor='black')
plt.xlabel("Number of interactions per item")
plt.ylabel("Number of items")
plt.title("Item popularity distribution")
plt.show()

# 텍스트 통계
counts = np.array(list(item_counts.values()))
print("Total items:", len(item_counts))
print("Min interactions per item:", counts.min())
print("Max interactions per item:", counts.max())
print("Mean interactions per item:", counts.mean())
print("Median interactions per item:", np.median(counts))

# 인기 아이템 TOP 10
top_items = item_counts.most_common(10)
print("Top 10 most popular items (item_id, count):")
for item, cnt in top_items:
    print(f"{item}: {cnt}")

In [ ]:
# 간단한 n-gram 통계
from collections import defaultdict
ngram_counts = defaultdict(int)
for seq in user_seqs_list:
    seq = [item for item in seq if item != book2idx[PAD_ID]]
    for i in range(len(seq)-1):
        ngram_counts[(seq[i], seq[i+1])] += 1

most_common_2grams = sorted(ngram_counts.items(), key=lambda x: -x[1])[:10]
print("Most common 2-item transitions:", most_common_2grams)

In [ ]:
# ===============================
# SASRec + TwoTower 모델
# ===============================
# sasrec_model = SASRec(hidden_dim=hidden_dim, max_len=seq_len) 위에서 학습한거 ㄱㄱ
two_tower = TwoTowerRecommender(
    sasrec_model=sasrec,
    e5_dim=e5_dim,
    hidden_dim=hidden_dim,
    proj_dim=proj_dim
).to(device)
optimizer = optim.Adam(two_tower.parameters(), lr=lr)

In [ ]:
"""
2. BPR fine-tuning
여기선 진짜 랭킹을 세우는 과정이라고 생각하면 된다
pretrain된 SASRec를 사용함!
https://chatgpt.com/share/690db409-f14c-800a-bf50-84385eb80753
"""
two_tower = TwoTowerRecommender(
    sasrec_model=sasrec_model,
    e5_dim=e5_dim,
    hidden_dim=hidden_dim,
    proj_dim=proj_dim
).to(device)
optimizer = optim.Adam(two_tower.parameters(), lr=lr)

book_emb_matrix = book_emb_matrix.to(device)

# BPR-style loss function
def bpr_loss(pos_scores, neg_scores):
    # pos_scores: [B], neg_scores: [B, n_neg]
    loss = -torch.log(torch.sigmoid(pos_scores.unsqueeze(1) - neg_scores)).mean()
    return loss

for epoch in range(epochs):
    two_tower.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}:'):
        user_seq, pos_emb = batch
        user_seq_embs = book_emb_matrix[user_seq]  # (B, L, 384)
        user_seq_embs = user_seq_embs.to(device)
        pos_emb = pos_emb.to(device)
        """
        # Negative 샘플링 (on the fly)
        neg_ids = torch.randint(0, book_emb_matrix.size(0), (user_seqs.size(0),))
        neg_book_embs = book_emb_matrix[neg_ids]
    
        모델 Forward
        pos_scores, neg_scores = model(user_seq_embs, pos_book_embs, neg_book_embs)
    
        손실 계산
        loss = -torch.log(torch.sigmoid(pos_scores - neg_scores)).mean()
        print("Loss:", loss.item())
        """
        # neg_embs = neg_embs.to(device)

        optimizer.zero_grad()
        # pos_scores, neg_scores = model(user_seq, pos_emb, neg_embs)
        # loss = bpr_loss(pos_scores, neg_scores)
        pos_scores = two_tower(user_seq_embs, pos_emb)
        loss = -torch.log(torch.sigmoid(pos_scores)).mean() 
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * user_seq.size(0)

    avg_loss = total_loss / len(dataset)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")